In [1]:
"""
COM7020 - MetroEnergy Solutions (MES) - Complete PoC Pipeline
================================================================
Runs end-to-end: dataset understanding -> data quality -> feature
engineering -> EDA -> modelling -> evaluation -> architecture diagram.

Requires:
    pip install pandas numpy matplotlib seaborn scikit-learn graphviz
    + the Graphviz system binary (apt/brew/installer - not just the
      Python package) for the final diagram section.

Expects metroenergy_daily_aggregated.csv in the same folder as this
script. All charts are saved into a ./figures folder as PNG files,
ready to drop into the Word report.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)


In [2]:

# ============================================================
# 1. UNDERSTANDING THE DATASET
# ============================================================
df = pd.read_csv("metroenergy_daily_aggregated.csv", parse_dates=["date"])

print("=" * 60)
print("1. DATASET OVERVIEW")
print("=" * 60)
print(df.head())
print("\nShape:", df.shape)
print("\nInfo:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nStatistical summary:")
print(df.describe())

1. DATASET OVERVIEW
       meter_id     region  has_solar  has_ev       date  \
0  MTR-AMS-1045  Amsterdam      False    True 2025-07-01   
1  MTR-AMS-1045  Amsterdam      False    True 2025-07-02   
2  MTR-AMS-1045  Amsterdam      False    True 2025-07-03   
3  MTR-AMS-1045  Amsterdam      False    True 2025-07-04   
4  MTR-AMS-1045  Amsterdam      False    True 2025-07-05   

   total_consumption_kwh  total_solar_kwh  total_ev_kwh  \
0                 41.386              0.0         8.696   
1                 40.748              0.0        12.633   
2                 41.252              0.0         0.000   
3                 40.888              0.0        10.309   
4                 42.273              0.0         9.342   

   peak_hour_consumption_kwh  avg_voltage_v  avg_temperature_c  fault_count  \
0                      3.456     230.112917          13.023750            0   
1                      3.602     230.044167          13.164583            0   
2                      3.15

In [3]:


# ============================================================
# 2. OUTLIER DETECTION (IQR method)
# ============================================================
print("\n" + "=" * 60)
print("2. OUTLIER DETECTION")
print("=" * 60)

num_cols = ["total_consumption_kwh", "peak_hour_consumption_kwh",
            "total_ev_kwh", "total_solar_kwh"]

fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df[col], ax=ax, color="#4C72B0")
    ax.set_title(col)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_outlier_boxplots.png")
plt.close()

def iqr_outlier_count(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

for col in num_cols:
    print(f"{col}: {iqr_outlier_count(df[col])} outliers (IQR method)")


2. OUTLIER DETECTION
total_consumption_kwh: 0 outliers (IQR method)
peak_hour_consumption_kwh: 2 outliers (IQR method)
total_ev_kwh: 486 outliers (IQR method)
total_solar_kwh: 0 outliers (IQR method)


In [4]:


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================
print("\n" + "=" * 60)
print("3. FEATURE ENGINEERING")
print("=" * 60)

df = df.sort_values(["meter_id", "date"]).reset_index(drop=True)

df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

grp = df.groupby("meter_id")
df["lag1_consumption"] = grp["total_consumption_kwh"].shift(1)
df["lag1_peak"] = grp["peak_hour_consumption_kwh"].shift(1)
df["rolling7_consumption"] = grp["total_consumption_kwh"].transform(
    lambda s: s.shift(1).rolling(7).mean()
)

# Next-day target: tomorrow's peak-hour consumption
df["target_next_peak"] = grp["peak_hour_consumption_kwh"].shift(-1)

df["has_solar"] = df["has_solar"].astype(int)
df["has_ev"] = df["has_ev"].astype(int)
df = pd.get_dummies(df, columns=["region"], prefix="region")

df_model = df.dropna(
    subset=["lag1_consumption", "lag1_peak", "rolling7_consumption", "target_next_peak"]
).copy()

print("Rows available for modelling:", df_model.shape[0], "of", df.shape[0])


3. FEATURE ENGINEERING
Rows available for modelling: 6720 of 7200


In [5]:


# ============================================================
# 4. EXPLORATORY DATA ANALYSIS
# ============================================================
print("\n" + "=" * 60)
print("4. EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# Consumption trend
daily_total = df.groupby("date")["total_consumption_kwh"].sum().reset_index()
plt.figure(figsize=(11, 4))
plt.plot(daily_total["date"], daily_total["total_consumption_kwh"], color="#4C72B0")
plt.title("Total Daily Consumption Across All Meters (Jul-Oct 2025)")
plt.xlabel("Date"); plt.ylabel("Total Consumption (kWh)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_consumption_trend.png")
plt.close()

# Regional comparison
region_cols = [c for c in df.columns if c.startswith("region_")]
region_avg = df.melt(id_vars=["total_consumption_kwh"], value_vars=region_cols,
                      var_name="region", value_name="is_region")
region_avg = region_avg[region_avg["is_region"] == 1]
region_avg["region"] = region_avg["region"].str.replace("region_", "")
plt.figure(figsize=(7, 4))
sns.boxplot(data=region_avg, x="region", y="total_consumption_kwh", palette="Set2")
plt.title("Daily Consumption by Region")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_regional_consumption.png")
plt.close()


4. EXPLORATORY DATA ANALYSIS


C:\Users\Awais\AppData\Local\Temp\ipykernel_51400\2148366045.py:25: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=region_avg, x="region", y="total_consumption_kwh", palette="Set2")


In [6]:

# Solar / EV comparison
solar_compare = df.groupby("has_solar")["total_consumption_kwh"].mean()
ev_compare = df.groupby("has_ev")["total_ev_kwh"].mean()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
solar_compare.plot(kind="bar", ax=axes[0], color=["#DD8452", "#55A868"])
axes[0].set_title("Avg Consumption: Solar vs Non-Solar Meters")
axes[0].set_xticklabels(["No Solar", "Has Solar"], rotation=0)
ev_compare.plot(kind="bar", ax=axes[1], color=["#DD8452", "#55A868"])
axes[1].set_title("Avg EV Load: EV vs Non-EV Meters")
axes[1].set_xticklabels(["No EV", "Has EV"], rotation=0)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_solar_ev_comparison.png")
plt.close()

In [7]:


# Correlation heatmap
corr_cols = ["total_consumption_kwh", "total_solar_kwh", "total_ev_kwh",
             "peak_hour_consumption_kwh", "avg_voltage_v", "avg_temperature_c",
             "fault_count", "total_net_load_kwh"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numerical Variables")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/05_correlation_heatmap.png")
plt.close()

In [8]:



# Fault events
fault_by_date = df.groupby("date")["fault_count"].sum()
plt.figure(figsize=(11, 3.5))
plt.plot(fault_by_date.index, fault_by_date.values, color="#C44E52")
plt.title("Daily Fault Events Across the Network")
plt.xlabel("Date"); plt.ylabel("Fault Count")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/06_fault_events.png")
plt.close()
print("Total fault events:", int(df["fault_count"].sum()), "out of", len(df), "meter-days")

Total fault events: 497 out of 7200 meter-days


In [9]:

# ============================================================
# 5. DATA PREPARATION FOR MODELLING (chronological split)
# ============================================================
print("\n" + "=" * 60)
print("5. DATA PREPARATION FOR MODELLING")
print("=" * 60)

feature_cols = ["lag1_consumption", "lag1_peak", "rolling7_consumption",
                 "avg_temperature_c", "avg_voltage_v", "day_of_week", "month",
                 "is_weekend", "has_solar", "has_ev"] + region_cols

split_date = df_model["date"].quantile(0.75)
train = df_model[df_model["date"] <= split_date]
test = df_model[df_model["date"] > split_date]

X_train, y_train = train[feature_cols], train["target_next_peak"]
X_test, y_test = test[feature_cols], test["target_next_peak"]
print("Split date:", split_date.date())
print("Train size:", X_train.shape, " Test size:", X_test.shape)


5. DATA PREPARATION FOR MODELLING
Split date: 2025-09-29
Train size: (5040, 14)  Test size: (1680, 14)


In [10]:


# ============================================================
# 6. MODEL DEVELOPMENT + HYPERPARAMETER TUNING
# ============================================================
print("\n" + "=" * 60)
print("6. MODEL DEVELOPMENT")
print("=" * 60)

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

results = []
for n_estimators in [100, 200]:
    for max_depth in [6, 10, None]:
        model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth,
                                       random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rmse = mean_squared_error(y_test, pred) ** 0.5
        results.append({"n_estimators": n_estimators, "max_depth": max_depth, "rmse": round(rmse, 4)})

tuning_df = pd.DataFrame(results).sort_values("rmse")
print(tuning_df)

best = tuning_df.iloc[0]
rf = RandomForestRegressor(
    n_estimators=int(best["n_estimators"]),
    max_depth=None if pd.isna(best["max_depth"]) else int(best["max_depth"]),
    random_state=42, n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("Selected configuration:", best.to_dict())


6. MODEL DEVELOPMENT
   n_estimators  max_depth    rmse
0           100        6.0  0.1450
3           200        6.0  0.1450
4           200       10.0  0.1467
1           100       10.0  0.1469
5           200        NaN  0.1491
2           100        NaN  0.1494
Selected configuration: {'n_estimators': 100.0, 'max_depth': 6.0, 'rmse': 0.145}


In [11]:
# ============================================================
# 7. MODEL EVALUATION
# ============================================================
print("\n" + "=" * 60)
print("7. MODEL EVALUATION")
print("=" * 60)

def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}")
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

metrics = [
    evaluate(y_test, lr_pred, "Linear Regression"),
    evaluate(y_test, rf_pred, "Random Forest (tuned)"),
]
metrics_df = pd.DataFrame(metrics)

plt.figure(figsize=(9, 4.5))
metrics_df.set_index("model")[["MAE", "RMSE"]].plot(kind="bar", ax=plt.gca(),
                                                       color=["#4C72B0", "#DD8452"])
plt.title("Model Comparison: MAE and RMSE")
plt.ylabel("kWh error")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/07_model_comparison.png")
plt.close()

sample = test.copy()
sample["predicted"] = rf_pred
daily_actual = sample.groupby("date")["target_next_peak"].mean()
daily_pred = sample.groupby("date")["predicted"].mean()

plt.figure(figsize=(11, 4.5))
plt.plot(daily_actual.index, daily_actual.values, label="Actual", color="#4C72B0")
plt.plot(daily_pred.index, daily_pred.values, label="Predicted", color="#DD8452", linestyle="--")
plt.title("Random Forest: Predicted vs Actual Next-Day Peak Demand (Test Period)")
plt.xlabel("Date"); plt.ylabel("Peak-Hour Consumption (kWh)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/08_predicted_vs_actual.png")
plt.close()

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
importances.plot(kind="barh", color="#55A868")
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/09_feature_importance.png")
plt.close()
print("\nFeature importances:\n", importances)



7. MODEL EVALUATION
Linear Regression: MAE=0.115  RMSE=0.144  R2=0.7643
Random Forest (tuned): MAE=0.116  RMSE=0.145  R2=0.7595

Feature importances:
 rolling7_consumption    0.925811
lag1_consumption        0.051984
avg_voltage_v           0.007273
lag1_peak               0.006277
avg_temperature_c       0.005924
day_of_week             0.001134
region_Manchester       0.000318
month                   0.000279
region_Amsterdam        0.000245
has_solar               0.000208
region_Berlin           0.000151
has_ev                  0.000149
is_weekend              0.000136
region_London           0.000109
dtype: float64


In [12]:
# If graphviz Python package isn't installed:
# !pip install graphviz
# Also needs the Graphviz system binary: apt install graphviz / brew install graphviz / Windows installer
!pip install graphviz
import graphviz
import os

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)